# H6: Gemini: универсальная VLM против специализированной OCR

**Проверяемая гипотеза:** На общих 200 test-примерах уступает PP-OCRv6 small не более чем на 2 п.п.

Мультимодальная генеративная модель получает изображение и инструкцию, возвращает текст. Это другой класс задачи декодирования, чем CTC. Точные слои, число параметров и состав обучения Gemini 3.8 Flash публично не раскрыты: внутреннюю архитектуру не выдумываем.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL_V2.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation_v2 import (manifest,run_local,run_gemini,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')


,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'gemini'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_01e828220fa7d238b62f51fbdc47409d,170370,data/benchmark/images/test/170370.01e828220fa7...
1,test_020a007c253ce7d1c28b683e15d7e83a,443368,data/benchmark/images/test/443368.020a007c253c...
2,test_034e8aac2ba9b9113823c98b4907b0b1,343212,data/benchmark/images/test/343212.034e8aac2ba9...
3,test_03727fc2067743a2ba7e9fefc0e2f33b,968856,data/benchmark/images/test/968856.03727fc20677...
4,test_040eb6388a6d6bf5fd5e7f6c9abbb0b3,338577,data/benchmark/images/test/338577.040eb6388a6d...
5,test_04e09ce988726e1cb165a646aeb34d4e,345475,data/benchmark/images/test/345475.04e09ce98872...
6,test_06696cbb2c68f7d23402e5b6a0639a72,531883,data/benchmark/images/test/531883.06696cbb2c68...
7,test_084df233edc69478564fa07cd73d20d1,020936,data/benchmark/images/test/020936.084df233edc6...
8,test_09e1357fae096888bb3b2325e29a79ea,577308,data/benchmark/images/test/577308.09e1357fae09...
9,test_0b071367e3cc92c8dd47d57bd81d8344,180332,data/benchmark/images/test/180332.0b071367e3cc...


Evaluation images: 200


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. Будет до 209 реальных API-запросов: 200 общего test и 9 target. Без повторов, concurrency=4; это расходует квоту.

In [3]:
frames=run_gemini()

gemini shared_200 20 / 200


gemini shared_200 40 / 200


gemini shared_200 60 / 200


gemini shared_200 80 / 200


gemini shared_200 100 / 200


gemini shared_200 120 / 200


gemini shared_200 140 / 200


gemini shared_200 160 / 200


gemini shared_200 180 / 200


gemini shared_200 200 / 200


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    destination=RESULTS/('gemini')/(dataset+'_scored.csv')
    joined.to_csv(destination,index=False)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))


shared_200 — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,test_01e828220fa7d238b62f51fbdc47409d,170370,\70370,GeminiError,Unexpected transcription format
10,test_0b84f116201e45db20ff1f0679382d18,370191,370t91,,
24,test_1b692807b54076d2a334baba1e3d13f6,685325,,GeminiError,Network/proxy connection failed; no direct fal...
36,test_2b7ac482ecd698adeedb9c94b7cce0c0,246624,,GeminiError,Network/proxy connection failed; no direct fal...
37,test_2c5466352b8c8d10f19d2bf8329323ec,951140,,GeminiError,Network/proxy connection failed; no direct fal...
43,test_3109dd415248747d9c0e3ef7a0e05616,366119,3661.19,GeminiError,Unexpected transcription format
53,test_3ba1ee92327f9732fee442ab37adb9a0,696398,456398,,
80,test_5ca83887857f79e8454600aa2dfadb1f,512690,,GeminiError,Network/proxy connection failed; no direct fal...
90,test_6c058c26bfb71f053ba15874bc35d7f6,621640,,GeminiError,Network/proxy connection failed; no direct fal...
91,test_6d3aff834435db6722280ded32983f13,174152,,GeminiError,Network/proxy connection failed; no direct fal...


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,,GeminiError,Network/proxy connection failed; no direct fal...
1,41ada5001f1e1e88,432077,,GeminiError,Network/proxy connection failed; no direct fal...
2,5cb6a434364bfe27,558796,,GeminiError,Network/proxy connection failed; no direct fal...
3,7619f1f1b372df6d,181263,,GeminiError,Network/proxy connection failed; no direct fal...
4,8731a1e555abd295,926325,,GeminiError,Network/proxy connection failed; no direct fal...
5,d4c0652824fb2dfd,639125,,GeminiError,Network/proxy connection failed; no direct fal...
6,d4d1163a8ce99293,642044,,GeminiError,Network/proxy connection failed; no direct fal...


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,shared_200,200,82,0.41,0.344157,0.479235,0.5825,116,0,5315.1634,19355.94928,12390.6236,21378.65524
1,target,7,0,0.00,0.000000,0.354330,1.0000,7,0,5328.2561,5332.22067,NaN,NaN


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('gemini')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H6')])
display(paired[paired.hypothesis.eq('H6')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
15,gemini,shared_200,200,82,0.41,0.344157,0.479235,0.5825,116,0,5315.1634,19355.94928,12390.6236,21378.65524
16,gemini,target,7,0,0.00,0.000000,0.354330,1.0000,7,0,5328.2561,5332.22067,NaN,NaN


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
5,H6,delta gemini-ppocr-small>=-2.00%,rejected,-0.53,-0.623811,-0.410215


,hypothesis,model_a,model_b,n,wins,losses,delta,delta_ci95_low,delta_ci95_high,mcnemar_exact_p,holm_p
4,H6,gemini,ppocr-small,200,6,112,-0.53,-0.623811,-0.410215,2.093279e-26,4.186558e-26
